# Training Deficits & Risk ClassificationConverted from `src/training_deficits.py`---

**Beschreibung:** Training Deficits - Identifies training needs and disciplinary actions

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Thresholds for classification
THRESHOLDS = {
    # YELLOW (Training recommended)
    'low_score': 2.5,           # Avg score below 2.5 → needs training
    'high_variance': 1.5,       # Std > 1.5 (inconsistent)
    'below_team_avg': -0.5,     # 0.5 points below team avg
    # RED (Disciplinary)
    'critical_low_score': 1.5,  # Avg score below 1.5
    'critical_min_score': 2.0,  # Min score below 2
}


def calculate_employee_metrics(scored_df):
    """Calculate performance metrics per employee.
    Returns: DataFrame with aggregated metrics
    """
    print("Calculating employee metrics...")
    
    # Only valid scores (assuming Q1 > 0 means answered/valid)
    valid_df = scored_df[scored_df['Q1'] > 0].copy()
    
    # Aggregation per assignee
    metrics = valid_df.groupby('assignee').agg({
        'Q1': ['mean', 'std', 'count', 'min'],
        'Q2': ['mean'],
        'Q3': ['mean']
    }).reset_index()
    
    # Flatten / rename multi-level columns
    metrics.columns = [
        'employee_id',
        'avg_q1', 'std_q1', 'ticket_count', 'min_q1',
        'avg_q2', 'avg_q3'
    ]
    
    # Q1 and Q2 both measure "Quality of Work", Q3 = "Client Relations"
    metrics['quality_score'] = (metrics['avg_q1'] + metrics['avg_q2']) / 2
    metrics['client_score']  = metrics['avg_q3']
    
    # Overall: 50% Quality + 50% Client Relations
    metrics['overall_score'] = (
        metrics['quality_score'] * 0.5 +
        metrics['client_score']  * 0.5
    )
    
    # Deviation from team average
    team_avg = metrics['overall_score'].mean()
    metrics['vs_team_avg'] = metrics['overall_score'] - team_avg
    
    print(f"   {len(metrics)} employees analyzed")
    print(f"   Team average: {team_avg:.2f}")
    
    return metrics


def classify_employee(metrics_row):
    """Classify an employee by risk level.
    Returns: dict with classification and recommendations
    """
    training_areas = []
    disciplinary_flags = []
    recommendations = []
    
    avg_score = metrics_row['overall_score']
    std_score = metrics_row['std_q1'] if pd.notna(metrics_row['std_q1']) else 0
    min_score = metrics_row['min_q1']
    vs_team   = metrics_row['vs_team_avg']
    
    # ─── TRAINING RECOMMENDED (YELLOW) ───
    if avg_score < THRESHOLDS['low_score']:
        training_areas.append('Solution Quality')
        recommendations.append('Workshop: Systematic Problem Analysis')
        
    if std_score > THRESHOLDS['high_variance']:
        training_areas.append('Consistency')
        recommendations.append('Coaching: Checklists for consistent quality')
        
    if vs_team < THRESHOLDS['below_team_avg']:
        training_areas.append('General Performance')
        recommendations.append('Mentoring: Pair work with experienced colleague')
    
    # ─── DISCIPLINARY (RED) ───
    if avg_score < THRESHOLDS['critical_low_score']:
        disciplinary_flags.append('Critically low performance')
        
    if min_score < THRESHOLDS['critical_min_score']:
        disciplinary_flags.append('Very poor individual ratings')
    
    # ─── RISK LEVEL ───
    if disciplinary_flags:
        risk_level = 'RED'
    elif training_areas:
        risk_level = 'YELLOW'
    else:
        risk_level = 'GREEN'
        recommendations.append('Good work! Keep it up.')
    
    return {
        'risk_level': risk_level,
        'training_areas': training_areas,
        'disciplinary_flags': disciplinary_flags,
        'recommendations': recommendations
    }


def analyze_all_employees(scored_df):
    """Analyze all employees and produce summary table."""
    print("\nTRAINING DEFICIT ANALYSIS")
    print("=" * 50)
    
    metrics_df = calculate_employee_metrics(scored_df)
    
    results = []
    for _, row in metrics_df.iterrows():
        classification = classify_employee(row)
        results.append({
            'employee':       row['employee_id'],
            'overall_score':  round(row['overall_score'], 2),
            'ticket_count':   int(row['ticket_count']),
            'risk_level':     classification['risk_level'],
            'training_areas': ', '.join(classification['training_areas']) or '-',
            'flags':          ', '.join(classification['disciplinary_flags']) or '-',
            'recommendations': '; '.join(classification['recommendations'])
        })
    
    results_df = pd.DataFrame(results)
    # Sort: RED first, then within each group by score ascending
    results_df = results_df.sort_values(['risk_level', 'overall_score'], ascending=[False, True])
    
    # Summary counts
    green  = (results_df['risk_level'] == 'GREEN').sum()
    yellow = (results_df['risk_level'] == 'YELLOW').sum()
    red    = (results_df['risk_level'] == 'RED').sum()
    
    print(f"\nRESULT:")
    print(f"   GREEN  (OK)        : {green}")
    print(f"   YELLOW (Training)  : {yellow}")
    print(f"   RED    (Disciplinary): {red}")
    
    return results_df


def print_training_plan(results_df):
    """Pretty-print prioritized training/disciplinary actions."""
    print("\n" + "=" * 50)
    print("TRAINING & ACTION PLAN")
    print("=" * 50)
    
    red_employees = results_df[results_df['risk_level'] == 'RED']
    if not red_employees.empty:
        print("\nURGENT – Immediate action required:")
        for _, emp in red_employees.iterrows():
            print(f"\n  {emp['employee']} (Score: {emp['overall_score']})")
            print(f"     Flags: {emp['flags']}")
    
    yellow_employees = results_df[results_df['risk_level'] == 'YELLOW']
    if not yellow_employees.empty:
        print("\nTRAINING RECOMMENDED (priority order):")
        for _, emp in yellow_employees.head(12).iterrows():   # limit to avoid flooding output
            print(f"\n  {emp['employee']} (Score: {emp['overall_score']})")
            print(f"     Areas: {emp['training_areas']}")
            print(f"     Actions: {emp['recommendations']}")
    
    if red_employees.empty and yellow_employees.empty:
        print("\nAll employees currently in GREEN zone – excellent!")

##  Execution

In [4]:
import pandas as pd
from pathlib import Path

print("=" * 50)
print(" TRAINING DEFICITS")
print("=" * 50)

# ─── Load the rated data ───
data_path = Path("data/raw/issues_snapshot_sample.xlsx")

if data_path.exists():
    scored_df = pd.read_excel(data_path)
    print(f" Loaded: {len(scored_df)} rated samples")
    
    # Run the full analysis
    results_df = analyze_all_employees(scored_df)
    
    # Show prioritized training/action plan
    print_training_plan(results_df)
    
    # Save results
    output_path = Path("reports/training_report.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(output_path, index=False)
    print(f"\n Saved: {output_path}")
    
else:
    print(" Rated samples not found!")
    print(f"Expected file: {data_path.absolute()}")

 TRAINING DEFICITS
 Rated samples not found!
Expected file: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/issues_snapshot_sample.xlsx
